# DHGNN + EBM trên 8 mã cổ phiếu
Một mô hình, 20 phiên tháng 11/2020 cho mỗi mã. Chọn tệp DHGNN_EBM_global8.zip khi dùng Colab. Gói chứa đặc trưng đã xử lý, checkpoint và kết quả đã đo.

In [ ]:
from pathlib import Path
import json, sys, subprocess, zipfile
import pandas as pd
from IPython.display import display, Image
project=Path.cwd()
if (project/"global_model"/"train_global.py").exists():
    project=project/"global_model"
if not (project/"train_global.py").exists():
    archives=list(Path.cwd().glob("DHGNN_EBM_global8.zip"))
    if not archives:
        from google.colab import files
        uploaded=files.upload()
        archives=[Path(name) for name in uploaded if name.endswith(".zip")]
    archive,=archives
    with zipfile.ZipFile(archive) as z:
        z.extractall(Path.cwd())
    project=Path.cwd()/"global_model"
out=project/"global8"
try:
    from compression import zstd
except ImportError:
    import importlib.util
    if importlib.util.find_spec("zstandard") is None:
        subprocess.check_call([sys.executable,"-m","pip","install","zstandard"])
assert (out/"model.pt").is_file()

## Dữ liệu, ngày chia và kết quả đã đo

In [ ]:
audit=pd.read_csv(out/"sessions.csv")
display(audit.groupby(["symbol","split"]).size().unstack(fill_value=0))
display(pd.read_csv(out/"metrics_by_symbol.csv"))
display(pd.read_csv(out/"viac_march_sessions.csv"))
display(Image(filename=str(out/"validation_and_event.png")))

## Huấn luyện lại
Đặt RUN_TRAIN=True để học lại 24 epoch trên dữ liệu của tám mã. Checkpoint được chọn từ 40 mã–phiên validation. Thiết bị CUDA được chọn tự động khi có.

In [ ]:
RUN_TRAIN = False
if RUN_TRAIN:
    subprocess.run([sys.executable,"-u",str(project/"train_global.py"),"--epochs","24"],check=True)
display(pd.DataFrame([json.loads((out/"metrics.json").read_text())["proxy_metrics"]]))
display(pd.read_csv(out/"history.csv"))